# Error Correcting Codes

## Hamming Distance

Explanation on [Hamming Distance](https://www.youtube.com/watch?v=kO6UlCY6idg) and invalid codewords

As explained in this[article](https://www.quora.com/What-is-the-hamming-distance-and-how-does-it-help-to-detect-errors-in-data-communication), if the hamming distance between any two distinct & valid codewords for a given code is D:

1. Up to D-1 errors can be detected, since any pattern of up to **D−1 bit flips** cannot transform one valid codeword into a different valid one, so the receiver can tell **something changed**. For example: a code with minimum distance 3 detects up to 2 bit errors.

2. The same minimum distance D lets the code correct up to **t = floor((D−1)/2)** bit errors by nearest-neighbor decoding: the received word is assigned to the closest codeword. If ≤ t errors occurred, the original codeword **remains the unique nearest codeword**, so that the original message can be recovered by choosing the nearest valid codeword. For example: distance-3 codes correct 1-bit errors.

In [ ]:
def get_ecc(hd):
	"""Gets the number of bits that can be corrected and detected for a given Hamming distance."""
	if hd < 1:
		return (0, 0)
	elif hd == 1:
		return (0, 1)
	elif hd == 2:
		return (0, 2)
	else:
		return (hd - 1) // 2, hd - 1

# Print a table of Hamming distances and their corresponding correctable and detectable bits
for hd in range(0, 10):
	correctable, detectable = get_ecc(hd)
	print(f"Hamming distance: {hd}, Correctable bits: {correctable}, Detectable bits: {detectable}")
	

Hamming distance: 0, Correctable bits: 0, Detectable bits: 0
Hamming distance: 1, Correctable bits: 0, Detectable bits: 1
Hamming distance: 2, Correctable bits: 0, Detectable bits: 2
Hamming distance: 3, Correctable bits: 1, Detectable bits: 2
Hamming distance: 4, Correctable bits: 1, Detectable bits: 3
Hamming distance: 5, Correctable bits: 2, Detectable bits: 4
Hamming distance: 6, Correctable bits: 2, Detectable bits: 5
Hamming distance: 7, Correctable bits: 3, Detectable bits: 6
Hamming distance: 8, Correctable bits: 3, Detectable bits: 7
Hamming distance: 9, Correctable bits: 4, Detectable bits: 8


In [19]:
# Find the number of parity bits needed for a given number of data bits to meet SEC property
def get_parity_bits(num_data_bits):
	"""Gets the number of parity bits needed for a given number of data bits to meet SEC property."""
	num_parity_bits = 0
	while (2 ** num_parity_bits) < (num_data_bits + num_parity_bits + 1):
		num_parity_bits += 1
	return num_parity_bits

# Print a table of data bits, and corresponding parity bits
for data_bits in [4, 6, 8, 11, 16, 32, 34, 64, 68, 128]:
	parity_bits = get_parity_bits(data_bits)
	print(f"Data bits: {data_bits}, Parity bits: {parity_bits}")

Data bits: 4, Parity bits: 3
Data bits: 6, Parity bits: 4
Data bits: 8, Parity bits: 4
Data bits: 11, Parity bits: 4
Data bits: 16, Parity bits: 5
Data bits: 32, Parity bits: 6
Data bits: 34, Parity bits: 6
Data bits: 64, Parity bits: 7
Data bits: 68, Parity bits: 7
Data bits: 128, Parity bits: 8


In [ ]:
# TODO: Find the number of parity bits needed to achieve a certain Hamming distance for a given number of data bits


# Generator Matrix

What are [Generator Matrices](https://www.youtube.com/watch?v=siMKiIFSmV0) in the context of ECC?

https://www.emergentmind.com/topics/ecc-secded

![](assets/hamming_matrix.png)

The generator matrix G can be used to determine the parity bits for a given data codeword (vector)

Message Vector M = (m1, m2, m3, ..., mk) of size 1 x k

Check bits (or parity) Vector C = (c1, c2, c3, ..., cm) of size 1 x m (in this video, they use g in place of m)

Overall Code Vector is concatenation of M and C, X = (M | C) of size 1 x n. Note that here, we're not considering the overall parity bit

X can be derived using, X = MG (matrix multiplication), where G is the generator matrix (of size k x n) defined as:

G = [Ik | P], where Ik is the k x k identity matrix, and P is the parity matrix of size k x m

The identity matrix simply copies the data bits to the final codeword, while the parity matrix calculates the check bits

In [8]:
# Write a function that outputs the codeword, given a data vector and generator matrix (numpy arrays)
import numpy as np
def encode(data, generator_matrix):
	"""Encodes the data using the generator matrix."""
	return np.dot(data, generator_matrix) % 2

# Write an example of how to use the encode function, for a data vector of length 4 and a generator matrix of size 4x7
# Generate a random data vector of length 4
data = np.random.randint(0, 2, size=4)  # Example data vector of length 4
print("Data vector:")
print(data)

# Write generator matrix as concatenation of identity matrix and parity matrix
parity_matrix = np.array([  [1, 1, 0],
							[1, 0, 1],
							[0, 1, 1],
							[1, 1, 1]])  # Example parity matrix of size 4x3
generator_matrix = np.concatenate((np.eye(4, dtype=int), parity_matrix), axis=1)  # Concatenate identity matrix and parity matrix to form generator matrix of size 4x7
print("\nGenerator matrix:")
print(generator_matrix)

codeword = encode(data, generator_matrix)
print("\nCodeword:")
print(codeword)

Data vector:
[1 1 1 0]

Generator matrix:
[[1 0 0 0 1 1 0]
 [0 1 0 0 1 0 1]
 [0 0 1 0 0 1 1]
 [0 0 0 1 1 1 1]]

Codeword:
[1 1 1 0 0 0 0]


# Parity-Check Matrix

> This section is currently WIP

H = [P' | Im] is of size (m x n)

where P' (size m x k) is transpose of P, and Im is identity matrix of size (m x m)

The decoding process constitutes

S = HR'

where

 * R' (size n x 1) is the transpose of received message vector R, and

 * S is the syndrome vector, which tells which parity bit detected the error

Each row of H is effectively the concatenation of **column of P** and **identify matrix**, so each element of S is vector multiplication, which is effectively selecting a subset of received vector R according to parity bit definition (essentially the same operation as used to generate parity bits)

1. If S = 0, no error is present, i.e., R = M

2. If S matches a column i of H, the ith bit in R has the error and can be corrected (by flipping)

OPEN: Understand why above and below

3. If S = hi ^ hj for 2 columns i and j in H, then a double bit error is detected

In [10]:
# Write a function to calculate the syndrome of a received codeword, given the parity-check matrix (numpy array)
def calculate_syndrome(received, parity_check_matrix):
	"""Calculates the syndrome of a received codeword using the parity-check matrix."""
	return np.dot(received, parity_check_matrix.T) % 2

# Write an example of how to use the calculate_syndrome function, for a received codeword of length 7 and a parity-check matrix of size 3x7
# Induce a single bit error in the codeword in a random position
received = codeword.copy()
random_position = np.random.randint(0, 7)
received[random_position] = (received[random_position] + 1) % 2  # Example received codeword with a single bit error at a random index
print("Received codeword:")
print(received)

# Write parity check matrix as concatenation of identity matrix and parity matrix
parity_check_matrix = np.concatenate((parity_matrix.T, np.eye(3, dtype=int)), axis=1)  # Concatenate parity matrix transpose and identity matrix to form parity-check matrix of size 3x7
print("\nParity-check matrix:")
print(parity_check_matrix)

syndrome = calculate_syndrome(received, parity_check_matrix)
print("\nSyndrome:")
print(syndrome)

Received codeword:
[0 1 1 0 0 0 0]

Parity-check matrix:
[[1 1 0 1 1 0 0]
 [1 0 1 1 0 1 0]
 [0 1 1 1 0 0 1]]

Syndrome:
[1 1 0]


In [12]:
# Write a function to correct the single bit error in the received codeword, given the syndrome and the parity-check matrix
def correct_error(received, syndrome, parity_check_matrix):
	"""Corrects a single bit error in the received codeword using the syndrome and parity-check matrix."""
	if np.count_nonzero(syndrome) == 0:
		return received  # No error detected
	else:
		# Compares each row of transposed parity-check matrix with the syndrome to find the error position
		# .all(axis=1) checks if all elements in the row match the syndrome, and np.where returns the indices of the rows that match
		# Picks the first matching row index as the error position and corrects the error by flipping the bit at that position in the received codeword
		error_position = np.where((parity_check_matrix.T == syndrome).all(axis=1))[0]
		if len(error_position) > 0:
			received[error_position[0]] = (received[error_position[0]] + 1) % 2  # Correct the error by flipping the bit at the error position
		return received
	
# Write an example of how to use the correct_error function, for a received codeword of length 7, a syndrome of length 3, and a parity-check matrix of size 3x7
corrected_codeword = correct_error(received, syndrome, parity_check_matrix)
print("\nCorrected codeword:")
print(corrected_codeword)


Corrected codeword:
[1 1 1 0 0 0 0]


# Double Error Detection

Overall parity bit addition

# Parity Matrix for Hamming Code

We note that the above expressions and code to create Generator and Parity Check matrices, and using them to identify parity bits, syndrome and correcting the detected single bit error are generic in nature, and do not depend on how the Parity Matrix is derived

Here, we study the algorithm used in original Hamming SECDED scheme

Based on 3B1B and GeeksforGeeks explanation, columns of P can be determined as follows:

1. Each column of P gets multiplied by the message vector bits, and constituents are added. This is same as using the column of P as **mask** to select the participating bits from the message vector to calculate the parity

2. Accordingly, for each parity bit p = 2^r where r in [0, m-1], generate the column r of matrix P such that

Pr (r-th column of P) = Vector with k values such that **j & 2^r != 0** for j in [0, k - 1]

The table below shows the Parity matrix for Hamming Code (11,4) as explained in 3B1B video

| Data Bit | Code Bit | 2^0 | 2^1 | 2^2 | 2^3 |
| --- | --- | --- | --- | --- | --- |
| 0 | 3 | 
| 1 | 5
| 2 | 6
| 3 | 7
| 4 | 9
| 5 | 10
| 6 | 11
| 7 | 12
| 8 | 13
| 9 | 14
| 10 | 15

The code below implements the algorithm to derive the Hamming Parity Matrix for a given data word size (k)

In [ ]:
# Write a function to create the Hamming code parity matrix, given the number of data bits
def create_hamming_parity_matrix(num_data_bits):
	"""Creates the parity matrix for a Hamming code given the number of data bits."""
	# The number of parity bits needed can be calculated using the formula: 2^p >= m + p + 1, where p is the number of parity bits and m is the number of data bits
	num_parity_bits = get_parity_bits(num_data_bits)

	# Initialize the parity matrix with zeros and fill it according to the Hamming code rules
	parity_matrix = np.zeros((num_data_bits, num_parity_bits), dtype=int)
	for i in range(num_data_bits):
		for j in range(num_parity_bits):
			# The parity bit at position 2^j covers the bits at positions that have the j-th bit set in their binary representation
			if (i + 1) & (2 ** j) != 0:
				parity_matrix[i][j] = 1
	return parity_matrix

# Write an example of how to use the create_hamming_parity_matrix function, for a number of data bits equal to 4
k = 4
h_parity_matrix = create_hamming_parity_matrix(num_data_bits=k)
print("Hamming Parity matrix:")
print(h_parity_matrix)

Parity matrix:
[[1 0 0]
 [0 1 0]
 [1 1 0]
 [0 0 1]]


# All 0's and 1's code words

For storage ECC, all 0's and all 1's have a [special significance](https://opentitan.org/book/hw/top_earlgrey/ip_autogen/flash_ctrl/doc/theory_of_operation.html#reliability-ecc:~:text=The%20ECC%20for%20flash%20is%20chosen%20such%20that%20a%20fully%20erased%20flash%20word%20has%20valid%20ECC.%20Likewise%20a%20flash%20word%20that%20is%20completely%200%20is%20also%20valid%20ECC.)

1. In Flash, when a sector is erased, all the bits are set, leading to all 1's stored. This means that when M = 'b1 (all 1's message), the valid parity bits nust also be all 1's

2. Since Flash doesn't support a single word erase, a common method used is to clear all the bits (including parity bits), so that the stored data is cleared. But such a resulting code word must also be valid, i.e., when all data bits are 0, the parity bits must also all be 0.

It is simple to see that from above equations to calculate parity bits, ensuring all 0's conditions is automatically obtained (since multiplication with a zero data vectors leads to an even parity bit of 0). 

Further, since for each parity bit P0 to P3, an odd number of data bits are combined, if all data bits are 1, the (even parity) calculation also ensures that all of the parity bits are also 1. Finally, in this specific case, since block size is even, there are odd number of bits, other than Pf, and which implies that Pf will also be 1 (if all other bits are 1)

To generalize that both all 0's & all 1's are valid codeword:

1. Even parity must be used for all of the parity as well as overall parity bit, so that all 0's data leads to all 0's parity bits

2. The parity check matrix (excluding the overall parity bit) H must have even number of 1's in every row, so that matrix multiplication with received vector R (all 1's) is zero for every row of H, i.e., all of the syndromes are 0 for R = all 1's

2. The value of k + m must be odd, so that overall parity bit is set to 1 when the data and parity bits are all 1's

In [ ]:
# Create an all 1's vector of k + m (both k and m are inputs to the function)
def create_all_ones_vector(num_data_bits, num_parity_bits):
	"""Creates an all 1's vector of length equal to the total number of bits (data + parity)."""
	total_bits = num_data_bits + num_parity_bits
	return np.ones(total_bits, dtype=int)

# Checks for all 0's and all 1's properties
def check_all_zeros_ones_properties(num_data_bits):
	"""Checks if the code can detect all 0's and all 1's errors."""
	num_parity_bits = get_parity_bits(num_data_bits)
	all_zeros_vector = np.zeros(num_data_bits + num_parity_bits, dtype=int)
	all_ones_vector = create_all_ones_vector(num_data_bits, num_parity_bits)
	print("All 0's vector:")
	print(all_zeros_vector)
	print("All 1's vector:")
	print(all_ones_vector)

	# Requirement 3: The value of k + m must be odd, so that overall parity bit is set to 1 when the data and parity bits are all 1's
	assert (num_data_bits + num_parity_bits) % 2 == 1, "The total number of bits (data + parity) must be odd to meet the all 1's property."

# Write an example of how to use the check_all_zeros_ones_properties function, for a number of data bits equal to 4
check_all_zeros_ones_properties(num_data_bits=34)

All 0's vector:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0]
All 1's vector:
[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1]


AssertionError: The total number of bits (data + parity) must be odd to meet the all 1's property.